# 🔥 Notebook 2: Cascading Failure — Breaker vs. No Breaker

Picture two services:

```
  clients ──► Service A ──► Service B (sick: slow + erroring)
```

B is slow. Every request to A blocks waiting for B. A's thread pool fills up. Now A itself looks dead to *its* callers. Congratulations: one small failure took down the whole stack. This is a **cascading failure**.

A circuit breaker between A and B turns that minute-long outage into a handful of fast errors. Let's measure it.

## 🛠️ Setup

```bash
cd 05-microservices/circuit-breaker
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

In [ ]:
import time, threading, statistics
from concurrent.futures import ThreadPoolExecutor

SLOW_B_DELAY = 1.0  # B hangs a full second before failing

def slow_b():
    time.sleep(SLOW_B_DELAY)
    raise RuntimeError('B timeout')

# --- A without a breaker: every call pays the full 1s timeout ---
def a_no_breaker():
    t0 = time.time()
    try:
        slow_b()
    except Exception:
        pass
    return time.time() - t0  # per-request latency

# --- A thread-safe breaker ---
class CB:
    def __init__(self, thr=5, reset=10.0):
        self.state = 'CLOSED'
        self.failures = 0
        self.opened_at = 0.0
        self.thr = thr
        self.reset = reset
        self.lock = threading.Lock()

    def call(self, fn):
        with self.lock:
            if self.state == 'OPEN':
                if time.time() - self.opened_at < self.reset:
                    raise RuntimeError('fast-fail: circuit OPEN')
                self.state = 'HALF_OPEN'
        try:
            r = fn()
            with self.lock:
                self.state = 'CLOSED'; self.failures = 0
            return r
        except Exception:
            with self.lock:
                self.failures += 1
                if self.state == 'HALF_OPEN' or self.failures >= self.thr:
                    self.state = 'OPEN'
                    self.opened_at = time.time()
            raise

cb = CB(thr=5, reset=10.0)

def a_with_breaker():
    t0 = time.time()
    try:
        cb.call(slow_b)
    except Exception:
        pass
    return time.time() - t0


### Simulate load

We use a **small worker pool** on purpose: in a real server, the pool is finite. When threads are stuck on B, the pool is what runs out.

In [ ]:
def run_load(handler, n=10, workers=2):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        latencies = list(ex.map(lambda _: handler(), range(n)))
    total = time.time() - t0
    return total, latencies

def report(label, total, lats):
    p50 = statistics.median(lats)
    p99 = sorted(lats)[int(0.99 * (len(lats)-1))]
    print(f'{label:14s} total={total:.2f}s  p50={p50*1000:7.1f}ms  p99={p99*1000:7.1f}ms  '
          f'max={max(lats)*1000:7.1f}ms')

print('B is sick (1s hang + error). 10 requests, 2 workers:\n')
total, lats = run_load(a_no_breaker,     n=10, workers=2)
report('no breaker',    total, lats)
total, lats = run_load(a_with_breaker,   n=10, workers=2)
report('with breaker',  total, lats)


**Reading the numbers:**
- *no breaker*: every request pays the full 1s timeout → high p50, high p99, workers stay stuck.
- *with breaker*: the first few requests pay the timeout, then the breaker trips. The rest fast-fail in microseconds. p99 still shows the pre-trip timeouts, but total wall time and worker occupancy drop dramatically.

In production that difference is the line between *one dependency has an issue* and *the entire site is down*.

## Bonus: add a fallback

Fast-failing is better than hanging, but a **fallback** is better than an error. Common fallbacks:
- cached value (even a bit stale),
- default / empty result,
- degraded feature ("recommendations unavailable" instead of 500).

The breaker stays the same; we just catch its fast-fail and return something useful.

In [ ]:
CACHE = {'recommendations': ['popular-item-1', 'popular-item-2']}
cb2 = CB(thr=3, reset=10.0)

def get_recommendations():
    try:
        return cb2.call(slow_b)  # would return fresh personalised recs
    except Exception:
        return CACHE['recommendations']  # graceful degradation

for i in range(6):
    t0 = time.time()
    result = get_recommendations()
    print(f'  req {i}: {time.time()-t0:5.2f}s  →  {result}')


### 🧠 Tips for production
- **Per-dependency breakers.** One breaker per downstream; don't share across services.
- **Fallbacks must be cheap.** Don't call another flaky service from your fallback.
- **Monitor state transitions.** Emit a metric every time the breaker opens/half-opens/closes. Dashboards love this.
- **Don't trip on user errors.** 4xx responses are the *caller's* fault; don't count them as failures.
- **Combine carefully with retries.** Retries inside a breaker are fine; retries *around* a breaker can hammer the probe. See Notebook 3.